# 03 — Contrôle qualité

**Objectif.** Rendre visibles les contrôles du quality gate : ce qui est vérifié, ce qui passe, ce qui échoue, et pourquoi.

**Source.** `data/processed/quality_report.json` et `pipeline_manifest.json`, produits par
`python -m france_business_dynamics.quality_gate`. Le notebook **ne recalcule aucun contrôle** : il lit le rapport
et illustre quelques problèmes en lisant les couches (fonctions de `validation/profile.py`).

## Comment lire PASS / WARN / FAIL

| Statut | Signification | Conséquence |
|---|---|---|
| **PASS** | Le contrôle est satisfait. | — |
| **WARN** | Anomalie de qualité *statistique* (ex. quelques codes commune absents du COG, dates illisibles). | Signalée, **ne bloque pas**. |
| **FAIL** | Violation d'*intégrité* (clé dupliquée, format invalide, réconciliation fausse, largeur SQL dépassée, section NAF Rev.2 sur une autre nomenclature). | **Bloque** : code de sortie 1, pas de chargement SQL. |
| **SKIPPED** | La couche n'existe pas encore (avec la commande pour la produire). | Ne bloque pas, sauf `--strict`. |

In [ ]:

import matplotlib.pyplot as plt
import polars as pl

from france_business_dynamics.utils.config import ProjectPaths

pl.Config.set_tbl_rows(25)
pl.Config.set_fmt_str_lengths(60)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

PATHS = ProjectPaths.from_env()
print("Répertoire de données :", PATHS.data)

from france_business_dynamics.utils.jsonio import read_json
from france_business_dynamics.validation import profile

report_path = PATHS.processed / "quality_report.json"
if not report_path.exists():
    raise RuntimeError("quality_report.json absent : exécuter `python -m france_business_dynamics.quality_gate`.")
report = read_json(report_path)
manifest_path = PATHS.processed / "pipeline_manifest.json"
manifest = read_json(manifest_path)["datasets"] if manifest_path.exists() else {}

print(f"Statut global : {report['status']}   |   mode : {report['mode']}   |   couches complètes : {report['complete']}")
print(f"Généré le {report['generated_at_utc']} (outil v{report['tool_version']})")
display(pl.DataFrame([{"indicateur": k, "valeur": str(v)} for k, v in report["summary"].items()]))

## 1. Périmètre : échantillon ou traitement complet ?

In [ ]:
rows = [{"couche": name, "statut": layer["status"], "lignes": layer.get("rows"), "périmètre": layer.get("scope", "—"),
         "construite_le": manifest.get(name, {}).get("built_at_utc", "—")} for name, layer in report["layers"].items()]
display(pl.DataFrame(rows))
if report["contains_samples"]:
    print("⚠ Ce rapport contient des échantillons : un PASS ici ne vaut pas validation du jeu complet.")

## 2. Résultat des contrôles par couche

In [ ]:
checks = pl.DataFrame([
    {"couche": name, **{k: str(c[k]) if k in ("observed", "expected") else c[k] for k in ("id", "description", "status", "severity", "observed", "expected")}}
    for name, layer in report["layers"].items() for c in layer.get("checks", [])
])
if checks.height:
    counts = checks.group_by("couche", "status").len().pivot(on="status", index="couche", values="len").fill_null(0)
    for status in ("PASS", "WARN", "FAIL"):
        if status not in counts.columns:
            counts = counts.with_columns(pl.lit(0).alias(status))
    order = {name: i for i, name in enumerate(report["layers"])}
    counts = counts.select("couche", "PASS", "WARN", "FAIL").sort(pl.col("couche").replace_strict(order, return_dtype=pl.Int32))
    display(counts)
    bottom = [0] * counts.height
    for status, color in (("PASS", "#27ae60"), ("WARN", "#f39c12"), ("FAIL", "#c0392b")):
        values = counts[status].to_list()
        plt.barh(counts["couche"].to_list(), values, left=bottom, label=status, color=color)
        bottom = [b + v for b, v in zip(bottom, values)]
    from matplotlib.ticker import MaxNLocator
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True)); plt.gca().invert_yaxis()
    plt.legend(); plt.title("Contrôles par couche et par statut"); plt.tight_layout(); plt.show()
else:
    print("Aucun contrôle exécuté.")

## 3. Contrôles en échec et avertissements (avec valeurs observées)

In [ ]:
problems = checks.filter(pl.col("status") != "PASS") if checks.height else checks
display(problems.select("couche", "status", "id", "description", "observed", "expected") if problems.height else "✅ Aucun échec ni avertissement.")

## 4. Tous les contrôles (référence)

In [ ]:
display(checks.select("couche", "id", "status", "severity", "description") if checks.height else "—")

## 5. Visualiser les problèmes dans les données

Les graphiques ci-dessous lisent la couche courante des établissements (`etablissement_current.parquet`).

In [ ]:
curated_path = PATHS.processed / "etablissement_current.parquet"
if curated_path.exists():
    curated = pl.scan_parquet(curated_path)
    geo = profile.value_distribution(curated, "geo_status")
    display(geo)
    plt.bar(geo["geo_status"].to_list(), geo["lignes"].to_list(), color=["#27ae60", "#f39c12", "#7f8c8d", "#c0392b"][: geo.height])
    plt.title("Appariement géographique au COG 2026 (MATCHED = commune trouvée)"); plt.yscale("log"); plt.show()

    nomenclature = profile.value_distribution(curated, "naf_nomenclature")
    display(nomenclature)

    invalid = curated.select(pl.col("date_creation_invalid").sum().alias("dates_invalides"), pl.len().alias("lignes")).collect()
    display(invalid)
else:
    print("etablissement_current.parquet absent — voir `transform`.")

## 6. Réconciliation : aucun établissement perdu entre la couche courante et le mart

La somme du mart doit être égale au nombre de lignes de la couche courante (limitée par `--limit` en échantillon).

In [ ]:
rec = [c for layer in report["layers"].values() for c in layer.get("checks", []) if c["id"].endswith(".reconciliation")]
display(pl.DataFrame([{"contrôle": c["id"], "statut": c["status"], "observé": str(c["observed"]), "attendu": c["expected"]} for c in rec]) if rec else "Aucune réconciliation exécutée.")